# 🛡️ ARAA — TRL GRPO Training on the Adversarial Reality Alignment Arena

This notebook trains a small LLM to act inside the **ARAA OpenEnv environment** using **TRL's GRPO trainer** with verifiable rewards.

**What happens:**
1. We clone the ARAA environment repo
2. Build a prompt dataset from ARAA observations
3. Define two independent reward functions (format + environment)
4. Train with `GRPOTrainer` so the model learns to output valid action vectors

**Runtime:** ~15-30 min on a free Colab T4 GPU

## 1. Install Dependencies

In [ ]:
!pip install -q openenv-core==0.2.3 trl>=0.18 transformers>=4.40 datasets>=2.18 accelerate>=0.30 peft>=0.11 sentencepiece>=0.2

## 2. Clone the ARAA Repository

Replace the URL below with your actual Hugging Face Space or GitHub repo URL.

In [ ]:
import os

REPO_URL = "https://huggingface.co/spaces/TEAM_SCALER/araa-adversarial-reality"  # UPDATE THIS
REPO_DIR = "araa-adversarial-reality"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

## 3. Imports & Configuration

In [ ]:
import torch
from transformers import AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

from env import ARAAEnv
from colab_trl_train import (
    build_dataset,
    env_reward_func,
    extract_reasoning_and_uncertainty,
    format_reward_func,
    generate_with_text_feedback,
    reasoning_reward_func,
)

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"

print(f"Model: {MODEL_NAME}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 4. Build the Prompt Dataset

We create prompts by resetting the ARAA environment with the `deceptive` preset.
Each prompt contains:
- Enterprise KPI board (10 named business metrics)
- Analyst report, adversarial memo, oversight alert
- Instructions to output a JSON list of 10 floats

In [ ]:
dataset = build_dataset(num_samples=64)
print(f"Dataset size: {len(dataset)}")
print(f"\n--- Example prompt (first 500 chars) ---")
print(dataset[0]["prompt"][:500])


## 5. Define Reward Functions

We use **three complementary reward channels** while keeping the notebook friendly for free Colab:

1. **Format reward** - checks whether the completion contains `FAULT ANALYSIS`, `UNCERTAINTY SCORE`, and a valid 10-value `ACTION VECTOR`
2. **Reasoning reward** - scores the human-readable analysis against the real environment state
3. **Environment reward** - steps the real ARAA environment and penalizes reward hacking via reward-gap and backdoor penalties

The key change: **every sample now also produces human-readable text feedback** internally. GRPO still needs numeric scores, so we keep those under the hood while surfacing readable reward explanations for debugging and self-repair.


In [ ]:
print("Reward functions ready")
print("  - format_reward_func: checks response structure")
print("  - reasoning_reward_func: scores human-readable fault analysis")
print("  - env_reward_func: uses true_reward + anti-hacking penalties")
print("  - text reward feedback is generated internally for clarity and repair")


## 6. Configure and Run GRPO Training

GRPO (Group Relative Policy Optimization) generates multiple completions per prompt,
scores them with the reward functions, and updates the model to favor higher-reward outputs.

This is the **RLVR** pattern: RL with Verifiable Rewards — no learned reward model needed.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

training_args = GRPOConfig(
    output_dir="outputs/trl_colab_run",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=192,
    num_train_epochs=1,
    logging_steps=1,
    save_strategy="no",
    report_to=[],
    use_cpu=not torch.cuda.is_available(),
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=[format_reward_func, reasoning_reward_func, env_reward_func],
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Trainer configured")
print("  - Reward functions: 3 (format + reasoning + environment)")
print("  - Text reward feedback: enabled")
print(f"  - Batch size: {training_args.per_device_train_batch_size}")
print(f"  - Generations per prompt: {training_args.num_generations}")
print(f"  - Learning rate: {training_args.learning_rate}")
print(f"  - Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")


In [ ]:
print("🚀 Starting GRPO training...")
trainer.train()
print("✅ Training complete!")

## 7. Save the Trained Model

In [ ]:
save_path = "outputs/trl_colab_run/final_model"
trainer.save_model(save_path)
print(f"✅ Model saved to {save_path}")

## 8. Quick Inference Test

Let's verify the trained model produces valid action vectors.

In [ ]:
from transformers import AutoModelForCausalLM

trained_model = AutoModelForCausalLM.from_pretrained(save_path)
trained_tokenizer = AutoTokenizer.from_pretrained(save_path)

test_env = ARAAEnv.from_preset("adversarial", seed=9999)
test_obs = test_env.reset(seed=9999, episode_id="inference-test")
test_prompt = test_env.build_llm_prompt(test_obs)

generated_text, result, action, feedback, repaired = generate_with_text_feedback(
    trained_model,
    trained_tokenizer,
    test_prompt,
    scenario="adversarial",
    seed=9999,
    max_new_tokens=192,
    allow_self_repair=True,
)
reasoning, uncertainty = extract_reasoning_and_uncertainty(generated_text)

print("--- Generated Output ---")
print(generated_text[:350])
print(f"\n--- Parsed Action Vector ---")
print(action)
print(f"\n--- Human-Readable Reward Feedback ---")
print(feedback.details)
print(f"\n--- Environment Result ---")
print(f"  Visible reward: {result.reward:.4f}")
print(f"  True reward:    {result.metadata['true_reward']:.4f}")
print(f"  Backdoor hit:   {result.metadata['backdoor_triggered']}")
print(f"  Attacked:       {result.metadata['attacked']}")
print(f"  Self-repair:    {'Applied once' if repaired else 'Not needed'}")
print(f"  Analysis:       {reasoning}")
print(f"  Uncertainty:    {uncertainty}")


## Summary

This notebook now demonstrates the updated ARAA GRPO loop with **human-readable text rewards**:

1. **Environment** -> ARAA provides structured prompts with deceptive enterprise KPIs
2. **Verifier** -> three reward channels score format, reasoning quality, and environment-grounded outcomes
3. **Text feedback** -> each sample also gets readable reward feedback so mistakes are understandable
4. **Trainer** -> TRL GRPO still optimizes numeric scores, which is required for the algorithm
5. **Inference** -> the trained model can read feedback, self-repair once, and then act in-environment

### Why this design works
- It satisfies GRPO's numeric optimization requirement
- It gives your mentor a clear human-readable reward layer
- It stays lightweight enough for free Colab by reusing the same shared helpers

---
*ARAA - Adversarial Reality Alignment Arena | OpenEnv Hackathon 2026 | Team Scaler*
